
# Walk-Forward Validation

The headline metrics elsewhere in this project come from a random split at a
single prediction date. That answers "can the model rank customers it has not
seen", but not "does it still work in March if it was trained in January",
which is the question that decides how often it needs retraining.

This notebook scores the model at five prediction dates a quarter apart, two
ways: retrained at each date, and trained once and left alone.


## 1. Setup

In [1]:

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from src.evaluation import bootstrap_metric, format_ci, paired_bootstrap
from src.features import PREDICTION_DATE, build_features
from src.generate_data import PREDICTION_DAY, START_DATE, expected_orders, generate
from src.scoring import (
    CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, NUMERIC_FEATURES,
    add_rate_features,
)

RANDOM_STATE = 42

def make_pipeline(numeric=None, categorical=CATEGORICAL_FEATURES, derive=True):
    steps = []
    if derive:
        steps.append(("rates", FunctionTransformer(add_rate_features)))
    transformers = [("num", Pipeline([
        ("i", SimpleImputer(strategy="median")), ("s", StandardScaler()),
    ]), list(numeric if numeric is not None else MODEL_NUMERIC_FEATURES))]
    if categorical:
        transformers.append(("cat", Pipeline([
            ("i", SimpleImputer(strategy="most_frequent")),
            ("o", OneHotEncoder(handle_unknown="ignore")),
        ]), list(categorical)))
    steps += [
        ("pre", ColumnTransformer(transformers)),
        ("model", GradientBoostingClassifier(
            random_state=RANDOM_STATE, learning_rate=0.03, max_depth=2,
            min_samples_leaf=10, n_estimators=100)),
    ]
    return Pipeline(steps)

customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")
events = pd.read_csv("../data/website_events.csv")
print("loaded")

PREDICTION_DAYS = [365, 456, 547, 638, 730]

def panel_at(day, customers, orders, events):
    date = START_DATE + pd.Timedelta(days=day)
    f = build_features(customers, orders, events, prediction_date=date)
    return date, f[NUMERIC_FEATURES + CATEGORICAL_FEATURES], f["churn"]


loaded


## 2. Retrained at each date: is the problem itself stable?

In [2]:

panels, rows = {}, []

for day in PREDICTION_DAYS:
    date, Xd, yd = panel_at(day, customers, orders, events)
    panels[day] = (date, Xd, yd)

    X_tr, X_te, y_tr, y_te = train_test_split(
        Xd, yd, test_size=0.30, stratify=yd, random_state=RANDOM_STATE)

    pipe = make_pipeline().fit(X_tr, y_tr)
    ci = bootstrap_metric(y_te.to_numpy(), pipe.predict_proba(X_te)[:, 1])

    rows.append({
        "date": str(date.date()), "n": len(Xd),
        "churn_rate": round(float(yd.mean()), 3),
        "auc": round(ci["estimate"], 4),
        "ci": f"[{ci['ci_lower']:.3f}, {ci['ci_upper']:.3f}]",
    })

pd.DataFrame(rows).set_index("date")


,n,churn_rate,auc,ci
date,,,,
2024-12-31,2139,0.644,0.6428,"[0.597, 0.687]"
2025-04-01,2750,0.653,0.6598,"[0.621, 0.698]"
2025-07-01,3290,0.643,0.6991,"[0.665, 0.732]"
2025-09-30,3848,0.635,0.6669,"[0.634, 0.699]"
2025-12-31,4418,0.646,0.6810,"[0.651, 0.711]"



Stable. The estimates wander by a few points and every interval overlaps every
other, so there is no evidence the problem gets harder or easier over the year.
The eligible population grows simply because more customers have passed the
30-day tenure requirement.

## 3. Trained once, never retrained: how fast does it go stale?


In [3]:

first = PREDICTION_DAYS[0]
_, X0, y0 = panels[first]
stale = make_pipeline().fit(X0, y0)

rows = []
for day in PREDICTION_DAYS:
    date, Xd, yd = panels[day]
    prob = stale.predict_proba(Xd)[:, 1]

    rows.append({
        "scored_at": str(date.date()),
        "months_old": round((day - first) / 30.44, 1),
        "auc": round(roc_auc_score(yd, prob), 4),
        "predicted_rate": round(float(prob.mean()), 3),
        "actual_rate": round(float(yd.mean()), 3),
        "calibration_error": round(float(prob.mean() - yd.mean()), 3),
    })

pd.DataFrame(rows).set_index("scored_at")


,months_old,auc,predicted_rate,actual_rate,calibration_error
scored_at,,,,,
2024-12-31,0.0,0.7034,0.644,0.644,-0.000
2025-04-01,3.0,0.6664,0.638,0.653,-0.015
2025-07-01,6.0,0.6797,0.630,0.643,-0.013
2025-09-30,9.0,0.6781,0.624,0.635,-0.011
2025-12-31,12.0,0.6770,0.617,0.646,-0.030



The first row is in-sample and therefore flattering; ignore it. What matters is
that a **twelve-month-old model has barely moved**.

That is a real property of this data, not a bug. Each customer's drift is a
fixed personal trait, so the *relationship* between observed history and future
behaviour never changes. A stationary process produces a model that does not
decay.

Which leaves an uncomfortable question: if this harness would report "all fine"
on a process that cannot break, how would we know it can detect a process that
does?

## 4. Testing the harness against a regime change

The generator accepts a population-wide shock. Here every customer's order rate
is halved from day 550 onward -- a macro event, a pricing change, a competitor
launch. The relationship the model learned is now wrong for everyone at once.


In [4]:

cs, os_, es, _ = generate(shock_day=550, shock_factor=0.5)

shocked = {}
for day in PREDICTION_DAYS:
    shocked[day] = panel_at(day, cs, os_, es)

_, Xs0, ys0 = shocked[first]
stale_shocked = make_pipeline().fit(Xs0, ys0)

rows = []
for day in PREDICTION_DAYS:
    date, Xd, yd = shocked[day]
    prob = stale_shocked.predict_proba(Xd)[:, 1]

    rows.append({
        "scored_at": str(date.date()),
        "months_old": round((day - first) / 30.44, 1),
        "auc": round(roc_auc_score(yd, prob), 4),
        "brier": round(brier_score_loss(yd, prob), 4),
        "predicted_rate": round(float(prob.mean()), 3),
        "actual_rate": round(float(yd.mean()), 3),
        "calibration_error": round(float(prob.mean() - yd.mean()), 3),
    })

pd.DataFrame(rows).set_index("scored_at")


,months_old,auc,brier,predicted_rate,actual_rate,calibration_error
scored_at,,,,,,
2024-12-31,0.0,0.7115,0.1991,0.655,0.655,-0.000
2025-04-01,3.0,0.6733,0.2058,0.668,0.662,0.006
2025-07-01,6.0,0.6717,0.1697,0.677,0.789,-0.112
2025-09-30,9.0,0.6636,0.1657,0.699,0.793,-0.095
2025-12-31,12.0,0.6543,0.1674,0.712,0.785,-0.073



## 5. The finding

The shock lands between the second and third rows, and the two metrics react
completely differently.

**ROC-AUC barely notices.** It drops by about two points across the whole
year -- well inside the noise seen in the stationary world. A monitor watching
discrimination alone would have raised nothing.

**Calibration breaks immediately.** Predicted churn stays near 0.68 while
actual churn jumps to 0.79. An eleven-point error, appearing in a single
quarter.

The reason is that ROC-AUC only measures *ordering*. Halving everyone's order
rate leaves the ranking almost untouched -- the same customers are still the
most likely to churn relative to each other -- while making every predicted
probability wrong in absolute terms. Any downstream decision using a
probability, which includes the cost-based threshold this project ships, is now
mis-set.

Worse, the **Brier score improves**, from about 0.20 to 0.17. It is a proper
scoring rule, but it is sensitive to the base rate, and a base rate moving
toward an extreme makes it easier to score well. Watched alone, it would have
looked like the model got better.

The practical conclusion: monitor predicted-versus-actual base rate. It is the
cheapest signal here and the only one that caught this.
